# The Test Set Is Not an Oracle

Article #33 examines what predictive evaluation can and cannot establish. Nine independent synthetic experiments cover iid CV, model search, nested CV, feature-selection leakage, temporal drift, look-ahead features, hierarchical groups, fold uncertainty, and adaptive test-set reuse.


## What is being estimated?

For a fixed learner A, cross-validation estimates a risk for a training procedure with approximately (K-1)/K of the available observations. Reusing those folds to select an algorithm makes their minimum an optimistic estimate of the selection process. Outer CV re-runs selection inside each training fold, but cannot make distribution shift or future-feature leakage disappear.

All observations here are synthetic; Brier loss is lower-is-better and every probability is evaluated against a simulated outcome.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile
import numpy as np
from IPython.display import Image, display
# Import from the article directory, repository root, or an explicit hint.
slug = 'the-test-set-is-not-an-oracle'
hint = os.environ.get('VALIDATION_ARTICLE_DIR')
location = Path.cwd().resolve()
candidates = ([Path(hint).expanduser().resolve()] if hint else
              [candidate for parent in (location, *location.parents)
               for candidate in (parent, parent/slug)])
study_dir = next((candidate for candidate in candidates
                  if (candidate/'analysis.py').is_file() and
                  (candidate/'the_test_set_is_not_an_oracle.ipynb').is_file()), None)
if study_dir is None:
    raise FileNotFoundError('Cannot find study; set VALIDATION_ARTICLE_DIR')
sys.path.insert(0, str(study_dir))
from analysis import Design, EXPERIMENTS, FIGURES, SEED, brier, run, stream
print('Seed:', SEED, '| experiments:', len(EXPERIMENTS))


In [ ]:
assert len(EXPERIMENTS) == len(FIGURES) == 9
labels = np.array([0, 1, 0, 1], dtype=np.int64)
assert abs(brier(labels, np.full(4, .5))-.25) < 1e-12
assert np.array_equal(stream(99, 'iid', 0).normal(size=6),
                      stream(99, 'iid', 0).normal(size=6))
print('Brier and random-stream identities passed')


## Execute all nine small simulations

This notebook uses three independent outer repetitions per experiment, compared with 20 to 70 for the archived full run. It writes a complete set of quick CSVs and figures in a temporary directory. Neither canonical evidence nor the random seed in the manuscript is altered. Error bars summarize Monte Carlo precision over repetitions, not the uncertainty of a real study.


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    destination = Path(directory)
    reports = run(Design(root=destination, quick=True))
    csv_files = sorted((destination/'results').glob('*_quick.csv'))
    png_files = sorted((destination/'figures').glob('*_quick.png'))
    assert len(csv_files) == 18 and len(png_files) == 9
    print('Tables:', len(csv_files), 'Figures:', len(png_files))
    for name, figure in zip(EXPERIMENTS, FIGURES):
        print('\n', name, '| summary cells:', len(reports[name]))
        display(Image(filename=str(destination/'figures'/f'{figure}_quick.png')))


## Boundaries of the examples

- A good iid CV estimate is conditional on a suitable data-generating and sampling process; it is not a transportability guarantee.
- Nested CV evaluates selection with smaller outer training sets and does not certify every model's eventual deployment risk.
- Feature selection on all labels and access to future features both leak information, through different mechanisms.
- A random fold of history is not necessarily an honest sample of the next period or a new site.
- Five folds are correlated through overlapping training sets; fold standard deviations are not independent replication errors.
- Exact public score queries are a hypothetical, intentionally vulnerable oracle, not a procedure to apply to real benchmarks.

When the kernel starts in an unrelated folder, define VALIDATION_ARTICLE_DIR as the absolute article path before starting the notebook. See the manuscript and study README for equations, sources, archival procedures, and limitations.
